# Pandas Complete Revision & Interview Practice Workbook
**Lectures 44–58 | Reference, moderate practice, project, challenging and company-style questions**

Run the notebook from top to bottom. Try each TODO cell before reading its solution. Examples use common, readable Pandas patterns; there can be multiple correct solutions.

## 1. Setup and sample data

In [ ]:
import pandas as pd
import numpy as np

employees = pd.DataFrame({
    "EmployeeID": [101,102,103,104,105,106,107,108],
    "Name": ["Amit","Priya","Rahul","Sneha","Neha","Karan","Isha","Vikram"],
    "Department": ["IT","HR","IT","Finance","HR","IT","Finance","HR"],
    "Age": [24,29,26,32,np.nan,28,31,25],
    "Salary": [45000,60000,35000,75000,50000,45000,82000,52000],
    "Performance": [85,92,78,95,88,85,91,73],
    "JoinDate": ["2024-01-10","2023-05-15","2024-07-20","2022-03-12",
                 "2023-09-01","2024-01-10","2021-11-05","2023-04-17"],
    "City": ["Pune","Mumbai","Pune","Delhi","Mumbai","Pune","Delhi","Mumbai"]
})
employees["JoinDate"] = pd.to_datetime(employees["JoinDate"])
employees

## 2. Pandas method reference

### Create, load, inspect, export
- `pd.Series(data)`, `pd.DataFrame(data)`
- `pd.read_csv("file.csv")`, `pd.read_excel("file.xlsx")`
- `df.head()`, `df.tail()`, `df.sample(n=3, random_state=42)`
- `df.shape`, `df.columns`, `df.index`, `df.dtypes`
- `df.info()`, `df.describe()`, `df.nunique()`
- `df.to_csv("output.csv", index=False)`, `df.to_excel("output.xlsx", index=False)`

### Select and modify
- One column: `df["Salary"]`; multiple: `df[["Name", "Salary"]]`
- Label selection: `df.loc[rows, columns]`; positional selection: `df.iloc[row_positions, column_positions]`
- Add column: `df["Bonus"] = df["Salary"] * 0.10`
- Drop: `df.drop(columns=["City"])`, `df.drop(index=[0])`
- Rename: `df.rename(columns={"Name": "EmployeeName"})`
- Index: `df.set_index("EmployeeID")`, `df.reset_index(drop=True)`

### Filter
- `df[df["Salary"] > 50000]`
- Combine conditions with `&` (AND), `|` (OR), `~` (NOT); put each condition in parentheses.
- `isin()`, `between()`, `query()`

### Sort, rank, top/bottom
- `sort_values("Salary", ascending=False)`, `sort_index()`
- `rank(ascending=False, method="dense")`; tie methods: `average`, `min`, `max`, `first`, `dense`
- `nlargest(5, "Salary")`, `nsmallest(5, "Salary")`
- `na_position="first"` / `"last"` for missing values during sorting

### Missing data and cleaning
- `isna()`, `isnull()`, `notna()`, `df.isna().sum()`
- `dropna(axis=0, how="any", subset=[...], thresh=...)`
- `fillna(value)`, fill with mean/median/mode, `ffill()`, `bfill()`
- `duplicated()`, `drop_duplicates(subset=..., keep="first")`
- `astype()`, `pd.to_numeric(errors="coerce")`, `replace()`
- Text: `.str.strip()`, `.str.title()`, `.str.lower()`, `.str.replace()`

### GroupBy and aggregation
- `groupby("Department")["Salary"].mean()`
- Aggregations: `sum`, `mean`, `median`, `min`, `max`, `count`, `size`, `std`, `nunique`, `agg`
- Named aggregation: `agg(AvgSalary=("Salary", "mean"))`
- `count()` counts non-missing values; `size()` counts rows.

### Combine tables
- `pd.merge(left, right, on="ID", how="inner")`; join types: `inner`, `left`, `right`, `outer`
- Different key names: `left_on=...`, `right_on=...`; suffixes: `suffixes=("_x", "_y")`
- `pd.concat([df1, df2], axis=0, ignore_index=True)` stacks rows
- `pd.concat([df1, df2], axis=1)` places columns side by side
- `.join()` commonly combines on index

### Summaries, functions, dates, strings, categories
- `pd.pivot_table(df, index=..., columns=..., values=..., aggfunc=..., fill_value=0, margins=True)`
- `pd.crosstab(df["A"], df["B"], normalize="index")`
- `Series.map()` for mapping values; `apply()` for applying functions; `lambda x: ...`
- Prefer vectorized operations when possible over row-wise `apply(axis=1)`.
- Dates: `pd.to_datetime(..., errors="coerce")`, `.dt.year`, `.dt.month`, `.dt.day_name()`, `.dt.quarter`, `.dt.to_period("M")`, `pd.date_range()`
- Strings: `.str.contains(..., case=False, na=False)`, `.str.startswith()`, `.str.endswith()`, `.str.split()`, `.str.extract()`, `.str.len()`
- Categories: `pd.cut(x, bins=..., labels=..., include_lowest=True)` uses specified ranges; `pd.qcut(x, q=4)` uses quantiles.
- Other: `value_counts()`, `unique()`, `nunique()`, `corr(numeric_only=True)`, `clip()`, `round()`, `memory_usage(deep=True)`, `read_csv(..., chunksize=10000)`

## 3. Examples of core operations

In [ ]:
# Select and filter
display(employees[["Name", "Department", "Salary"]])
display(employees.loc[employees["Department"] == "IT", ["Name", "Salary"]])
display(employees[(employees["Salary"] >= 45000) & (employees["Performance"] >= 85)])
display(employees[employees["Department"].isin(["HR", "Finance"])])

# Sort and rank
display(employees.sort_values(["Department", "Salary"], ascending=[True, False]))
ranked = employees.copy()
ranked["SalaryRank"] = ranked["Salary"].rank(ascending=False, method="dense")
display(ranked[["Name", "Salary", "SalaryRank"]])

# Missing values and text cleaning
dirty = pd.DataFrame({"Name":[" Amit ","PRIYA","Rahul","Rahul"],
                      "Age":["24","unknown","26","26"],
                      "City":["PUNE","Mumbai ",None,None],
                      "Salary":[45000,np.nan,35000,35000]})
clean = dirty.copy()
clean["Name"] = clean["Name"].str.strip().str.title()
clean["City"] = clean["City"].str.strip().str.title()
clean["Age"] = pd.to_numeric(clean["Age"], errors="coerce")
clean["Salary"] = clean["Salary"].fillna(clean["Salary"].median())
clean = clean.drop_duplicates()
display(clean)

# Group summaries
display(employees.groupby("Department").agg(
    EmployeeCount=("EmployeeID","count"),
    AverageSalary=("Salary","mean"),
    HighestSalary=("Salary","max"),
    AveragePerformance=("Performance","mean")
).round(2))

# Merge and pivot
departments = pd.DataFrame({"Department":["IT","HR","Finance"],
                            "Manager":["Meera","Anil","Farah"]})
display(pd.merge(employees, departments, on="Department", how="left"))
display(pd.pivot_table(employees, index="Department", columns="City",
                       values="Salary", aggfunc="mean", fill_value=0))
display(pd.crosstab(employees["Department"], employees["City"], margins=True))

# Apply/map and dates
display(employees.assign(Bonus=employees["Salary"] * 0.10))
display(employees.assign(JoinYear=employees["JoinDate"].dt.year,
                         JoinMonth=employees["JoinDate"].dt.month_name()))
print(employees["Department"].value_counts())

# Part A — Moderate practice questions

Try each TODO cell yourself before running the solution below it.

### Moderate 1. Filtering\nDisplay IT employees whose salary is greater than 40,000.

In [ ]:
# TODO: Write your solution

**Solution (check after trying):**

In [ ]:
employees[(employees["Department"] == "IT") & (employees["Salary"] > 40000)]

### Moderate 2. Missing values\nCopy the DataFrame and fill missing Age values with the median.

In [ ]:
# TODO: Write your solution

**Solution (check after trying):**

In [ ]:
temp = employees.copy()
temp["Age"] = temp["Age"].fillna(temp["Age"].median())
temp[["Name", "Age"]]

### Moderate 3. GroupBy\nCalculate employee count and average salary for each department.

In [ ]:
# TODO: Write your solution

**Solution (check after trying):**

In [ ]:
employees.groupby("Department").agg(EmployeeCount=("EmployeeID","count"), AverageSalary=("Salary","mean"))

### Moderate 4. Top records\nDisplay the two highest-paid employees.

In [ ]:
# TODO: Write your solution

**Solution (check after trying):**

In [ ]:
employees.nlargest(2, "Salary")[["Name", "Salary"]]

### Moderate 5. String cleaning\nClean spaces and capitalization in [" Pune ", "MUMBAI", "delhi "].

In [ ]:
# TODO: Write your solution

**Solution (check after trying):**

In [ ]:
cities = pd.Series([" Pune ", "MUMBAI", "delhi "])
cities.str.strip().str.title()

### Moderate 6. Left merge\nMerge customer and order tables while keeping every customer.

In [ ]:
# TODO: Write your solution

**Solution (check after trying):**

In [ ]:
customers = pd.DataFrame({"CustomerID":[1,2], "Name":["Asha","Dev"]})
orders = pd.DataFrame({"CustomerID":[1,1,3], "Amount":[500,700,300]})
pd.merge(customers, orders, on="CustomerID", how="left")

### Moderate 7. Date extraction\nConvert Date to datetime and extract its year.

In [ ]:
# TODO: Write your solution

**Solution (check after trying):**

In [ ]:
dates = pd.DataFrame({"Date":["2024-01-10","2023-08-12"]})
dates["Date"] = pd.to_datetime(dates["Date"], errors="coerce")
dates["Year"] = dates["Date"].dt.year
dates

### Moderate 8. Pivot table\nCalculate average salary by Department and City.

In [ ]:
# TODO: Write your solution

**Solution (check after trying):**

In [ ]:
pd.pivot_table(employees, index="Department", columns="City", values="Salary", aggfunc="mean")

# Part B — End-to-end data cleaning and analysis project

Tasks: standardize names and cities, safely convert Amount to numeric, parse dates, remove duplicates, fill missing Amount using the median, summarize revenue by city and month, and export the cleaned data.

In [ ]:
raw_sales = pd.DataFrame({
    "OrderID":[1,2,3,4,4,6,7],
    "Customer":[" Amit ","PRIYA","Rahul","Sneha","Sneha","Neha",None],
    "City":["PUNE","Mumbai ","pune","Delhi","Delhi",None,"Mumbai"],
    "Amount":["1200","2500","unknown","4000","4000","1800","2200"],
    "OrderDate":["2024-01-03","2024-01-04","bad-date","2024-02-01",
                 "2024-02-01","2024-02-10","2024-03-12"]
})
sales_clean = raw_sales.copy()
sales_clean["Customer"] = sales_clean["Customer"].str.strip().str.title()
sales_clean["City"] = sales_clean["City"].str.strip().str.title()
sales_clean["Amount"] = pd.to_numeric(sales_clean["Amount"], errors="coerce")
sales_clean["OrderDate"] = pd.to_datetime(sales_clean["OrderDate"], errors="coerce")
sales_clean = sales_clean.drop_duplicates()
sales_clean["Amount"] = sales_clean["Amount"].fillna(sales_clean["Amount"].median())

display(sales_clean)
display(sales_clean.groupby("City", dropna=False)["Amount"].sum().sort_values(ascending=False))
monthly = (sales_clean.dropna(subset=["OrderDate"])
           .assign(Month=lambda x: x["OrderDate"].dt.to_period("M"))
           .groupby("Month")["Amount"].sum())
display(monthly)
sales_clean.to_csv("sales_clean.csv", index=False)

# Part C — Challenging questions

Attempt each one before running its solution. These combine several Pandas skills.

### Challenge 1. Department-wise highest salary, including ties\nReturn all employees whose salary equals the maximum salary in their department.

In [ ]:
# TODO: Write your solution

**Solution:**

In [ ]:
max_salary = employees.groupby("Department")["Salary"].transform("max")
employees[employees["Salary"] == max_salary][["Name","Department","Salary"]]

### Challenge 2. Second-highest distinct salary\nReturn the second-highest distinct salary; handle the case where fewer than two distinct salaries exist.

In [ ]:
# TODO: Write your solution

**Solution:**

In [ ]:
salaries = employees["Salary"].dropna().drop_duplicates().sort_values(ascending=False)
second_highest = salaries.iloc[1] if len(salaries) >= 2 else None
second_highest

### Challenge 3. Above-department-average salaries\nReturn employees earning more than their department's average salary.

In [ ]:
# TODO: Write your solution

**Solution:**

In [ ]:
dept_avg = employees.groupby("Department")["Salary"].transform("mean")
employees[employees["Salary"] > dept_avg]

### Challenge 4. Rank salary within department\nHighest salary gets rank 1; ties share rank and ranks have no gaps.

In [ ]:
# TODO: Write your solution

**Solution:**

In [ ]:
ranked = employees.copy()
ranked["DeptRank"] = ranked.groupby("Department")["Salary"].rank(ascending=False, method="dense")
ranked[["Name","Department","Salary","DeptRank"]].sort_values(["Department","DeptRank"])

### Challenge 5. Duplicate business keys\nReturn all records where OrderID occurs more than once.

In [ ]:
# TODO: Write your solution

**Solution:**

In [ ]:
raw_sales[raw_sales.duplicated(subset=["OrderID"], keep=False)].sort_values("OrderID")

### Challenge 6. Department headcount percentages\nCalculate each department's share of all employees.

In [ ]:
# TODO: Write your solution

**Solution:**

In [ ]:
counts = employees.groupby("Department").size().rename("Count").to_frame()
counts["Percent"] = counts["Count"] / counts["Count"].sum() * 100
counts.round(2)

### Challenge 7. Customer spend, including customers without orders\nMerge customers and orders, then calculate total spend and order count for every customer.

In [ ]:
# TODO: Write your solution

**Solution:**

In [ ]:
customers = pd.DataFrame({"CustomerID":[1,2,3], "Customer":["Asha","Dev","Mina"]})
orders = pd.DataFrame({"OrderID":[10,11,12], "CustomerID":[1,1,2], "Amount":[500,300,900]})
merged = customers.merge(orders, on="CustomerID", how="left")
merged.groupby(["CustomerID","Customer"], as_index=False).agg(TotalSpend=("Amount","sum"), OrderCount=("OrderID","count"))

### Challenge 8. Monthly revenue with invalid dates excluded\nUse sales_clean; ignore rows where OrderDate could not be parsed.

In [ ]:
# TODO: Write your solution

**Solution:**

In [ ]:
sales_clean.dropna(subset=["OrderDate"]).assign(Month=lambda x: x["OrderDate"].dt.to_period("M")).groupby("Month", as_index=False)["Amount"].sum()

# Part D — Company-style interview patterns

These are representative junior analyst/Python screening patterns, not a guarantee that every company asks these exact questions.

## 1. Top 3 salaries in each department

In [ ]:
employees.sort_values(["Department","Salary"], ascending=[True,False]).groupby("Department").head(3)

## 2. Most frequent value, including ties

In [ ]:
sample = pd.DataFrame({"City":["Pune","Delhi","Pune","Mumbai","Pune","Delhi"]})
counts = sample["City"].value_counts()
counts[counts == counts.max()]

## 3. Keep latest record per customer

In [ ]:
transactions = pd.DataFrame({
    "CustomerID":[1,1,2,2,2],
    "Date":pd.to_datetime(["2024-01-01","2024-03-01","2024-02-01","2024-01-10","2024-04-01"]),
    "Amount":[100,150,200,180,250]
})
transactions.sort_values("Date").drop_duplicates("CustomerID", keep="last")

## 4. Cumulative sales over time

In [ ]:
daily = pd.DataFrame({"Date":pd.to_datetime(["2024-01-01","2024-01-02","2024-01-03","2024-01-04"]),
                      "Sales":[100,200,150,300]}).sort_values("Date")
daily["CumulativeSales"] = daily["Sales"].cumsum()
daily

## 5. Percent of total

In [ ]:
sales = pd.DataFrame({"Product":["A","B","C"], "Sales":[100,300,600]})
sales["PercentOfTotal"] = sales["Sales"] / sales["Sales"].sum() * 100
sales

## 6. Find unmatched keys after a merge

In [ ]:
left = pd.DataFrame({"ID":[1,2,3]})
right = pd.DataFrame({"ID":[2,3,4], "Value":["B","C","D"]})
checked = left.merge(right, on="ID", how="left", indicator=True)
display(checked)
checked[checked["_merge"] == "left_only"]

## 7. Validate merge keys before joining\nDuplicate keys can unexpectedly multiply rows in a merge. Check uniqueness and use `validate=` when the relationship is known.

In [ ]:
customers = pd.DataFrame({"CustomerID":[1,2], "Name":["Asha","Dev"]})
orders = pd.DataFrame({"OrderID":[10,11,12], "CustomerID":[1,1,2]})
orders.merge(customers, on="CustomerID", how="left", validate="many_to_one")

# Part E — Rapid interview knowledge check

Try to answer without running code:
1. Difference between `count()` and `size()` in a groupby?
2. Why use parentheses around conditions combined with `&` and `|`?
3. When is a left merge preferable to an inner merge?
4. Why use `pd.to_numeric(..., errors="coerce")` on messy input?
5. Difference between `cut()` and `qcut()`?
6. Why is vectorization often preferable to `apply(axis=1)`?
7. Difference between `loc` and `iloc`?
8. What should you consider before filling missing numeric values with a mean?
9. Why validate key uniqueness before merging?
10. How can `_merge` from `indicator=True` help identify unmatched rows?

# Final readiness checklist
- [ ] Load and inspect files.
- [ ] Select, filter, sort, and rank records.
- [ ] Clean text, types, missing values, and duplicates.
- [ ] Group and aggregate.
- [ ] Merge and concatenate tables safely.
- [ ] Create pivot tables and crosstabs.
- [ ] Work with strings and dates.
- [ ] Categorize with `cut()` and `qcut()`.
- [ ] Complete the project and challenging questions independently.
- [ ] Explain your approach clearly in an interview.

## Next lecture
**Lecture 59 — Matplotlib Introduction and Line Charts.**